In [ ]:

!pip install medmnist

import torch
import torch.nn as nn
import torch.optim as optim
import time
from medmnist import PneumoniaMNIST
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")


if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

#
train_dataset = PneumoniaMNIST(split='train', transform=data_transform, download=True)
test_dataset = PneumoniaMNIST(split='test', transform=data_transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=256, shuffle=False)

class MedicalCNN(nn.Module):
    def __init__(self):
        super(MedicalCNN, self).__init__()
        # Layer 1
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        # Layer 2
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        # Fully Connected Layer
        self.fc = nn.Linear(32 * 7 * 7, 1)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1) # Flatten
        out = self.fc(out)
        return out


model = MedicalCNN().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


print("\nStarting GPU Training...")
start_time = time.time()

for epoch in range(5):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        # MOVE DATA TO GPU (Crucial Step 2)
        images = images.to(device)
        labels = labels.to(device).float()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # average loss for this epoch
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/5], Loss: {avg_loss:.4f}")

end_time = time.time()
print(f"\nTotal Training Time: {end_time - start_time:.2f} seconds")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.7 MB/s eta 0:00:00
Training on: cuda
GPU Name: Tesla T4


100%|██████████| 4.17M/4.17M [00:01<00:00, 3.44MB/s]



Starting GPU Training...
Epoch [1/5], Loss: 0.3161
Epoch [2/5], Loss: 0.1409
Epoch [3/5], Loss: 0.1115
Epoch [4/5], Loss: 0.0975
Epoch [5/5], Loss: 0.0927

Total Training Time: 5.98 seconds
